# Day 7 Lab 08: CDC Latest Order State

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_08_cdc_latest_order_state.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read valid Silver order events.
- Explicitly deduplicate duplicate source events before CDC.
- Use a window function helper to select the latest state per order.
- Write and preview the current-state Silver table.

**Dependency:** Run Lab/Notebook 07 first so `lake/silver/orders_valid` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [4]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [5]:

import importlib
import day7_common
day7_common = importlib.reload(day7_common)

from day7_common import LAKE_DIR, OUTPUT_DIR, deduplicate_order_events, ensure_output_dirs, latest_order_state, require_source_data, spark_session, write_csv_dir, read_parquet, write_parquet

## 2. Start Spark and Read Valid Orders

In [6]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook08CdcLatestState")

valid = read_parquet(spark, LAKE_DIR / "silver" / "orders_valid")
valid.select("order_id", "event_id", "status", "event_time_ts").orderBy("order_id", "event_time_ts").show(30, truncate=False)

+--------+--------+---------+-------------------+
|order_id|event_id|status   |event_time_ts      |
+--------+--------+---------+-------------------+
|1001    |evt-1001|NEW      |2026-05-29T07:30:01|
|1001    |evt-1007|SHIPPED  |2026-05-30T14:30:01|
|1002    |evt-1002|NEW      |2026-05-29T07:30:05|
|1002    |evt-1004|NEW      |2026-05-29T07:30:05|
|1002    |evt-1011|CANCELLED|2026-05-30T14:42:00|
|1003    |evt-1003|NEW      |2026-05-29T07:30:08|
|1004    |evt-1005|CANCELLED|2026-05-29T07:30:20|
|1006    |evt-1008|NEW      |2026-05-30T14:35:00|
|1007    |evt-1009|NEW      |2026-05-30T14:37:00|
|1009    |evt-1012|PAID     |2026-05-30T14:45:00|
+--------+--------+---------+-------------------+



## 3. Deduplicate Valid Events

In [7]:
deduplicated = deduplicate_order_events(valid)
dedup_preview = deduplicated.select("order_id", "event_id", "status", "amount", "currency", "event_time_ts").orderBy("order_id", "event_time_ts", "event_id")
dedup_preview.show(30, truncate=False)
print(f"Valid rows before deduplication: {valid.count()}")
print(f"Rows after explicit Silver deduplication: {deduplicated.count()}")

+--------+--------+---------+-------+--------+-------------------+
|order_id|event_id|status   |amount |currency|event_time_ts      |
+--------+--------+---------+-------+--------+-------------------+
|1001    |evt-1001|NEW      |1299.99|USD     |2026-05-29T07:30:01|
|1001    |evt-1007|SHIPPED  |1299.99|USD     |2026-05-30T14:30:01|
|1002    |evt-1002|NEW      |799.0  |USD     |2026-05-29T07:30:05|
|1002    |evt-1011|CANCELLED|799.0  |USD     |2026-05-30T14:42:00|
|1003    |evt-1003|NEW      |149.5  |USD     |2026-05-29T07:30:08|
|1004    |evt-1005|CANCELLED|450.0  |USD     |2026-05-29T07:30:20|
|1006    |evt-1008|NEW      |89.99  |EUR     |2026-05-30T14:35:00|
|1007    |evt-1009|NEW      |450.0  |USD     |2026-05-30T14:37:00|
|1009    |evt-1012|PAID     |40.0   |USD     |2026-05-30T14:45:00|
+--------+--------+---------+-------+--------+-------------------+

Valid rows before deduplication: 10
Rows after explicit Silver deduplication: 9


## 4. Calculate and Inspect Current State

In [8]:
current = latest_order_state(deduplicated)
preview = current.select("order_id", "event_id", "status", "amount", "currency", "event_time_ts").orderBy("order_id")
preview.show(20, truncate=False)

+--------+--------+---------+-------+--------+-------------------+
|order_id|event_id|status   |amount |currency|event_time_ts      |
+--------+--------+---------+-------+--------+-------------------+
|1001    |evt-1007|SHIPPED  |1299.99|USD     |2026-05-30T14:30:01|
|1002    |evt-1011|CANCELLED|799.0  |USD     |2026-05-30T14:42:00|
|1003    |evt-1003|NEW      |149.5  |USD     |2026-05-29T07:30:08|
|1004    |evt-1005|CANCELLED|450.0  |USD     |2026-05-29T07:30:20|
|1006    |evt-1008|NEW      |89.99  |EUR     |2026-05-30T14:35:00|
|1007    |evt-1009|NEW      |450.0  |USD     |2026-05-30T14:37:00|
|1009    |evt-1012|PAID     |40.0   |USD     |2026-05-30T14:45:00|
+--------+--------+---------+-------+--------+-------------------+



## 5. Write Current-State Silver Table

In [9]:
write_parquet(deduplicated, LAKE_DIR / "silver" / "orders_deduplicated", mode="overwrite")
current_path = LAKE_DIR / "silver" / "orders_current"
write_parquet(current, current_path, mode="overwrite")
write_csv_dir(dedup_preview, OUTPUT_DIR / "notebook_08_deduplicated_events_preview")
write_csv_dir(preview, OUTPUT_DIR / "notebook_08_orders_current_preview")
print(f"Current order rows: {current.count()}")

Current order rows: 7


## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [10]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()